# Patient Drug Event Analysis

Run cells in order. This is retrospective report classification using your merged CSV. The notebook includes the complete source and requires no repository clone. Unknown outcome codes are excluded from training. Measured results are produced from your uploaded data; the guide's historical scores are not copied into the evaluation.

In [ ]:
%pip -q install pandas==2.2.3 numpy==2.2.6 matplotlib==3.10.3 scikit-learn==1.6.1 imbalanced-learn==0.13.0 joblib==1.5.1

## Create the project

This cell writes the same shared Python module used locally. Restart the runtime after package installation if prompted, then continue.

In [ ]:
from pathlib import Path
import sys, json
BASE = Path("/content/patient-drug-event-analysis")
BASE.mkdir(parents=True, exist_ok=True)
SOURCES = {'src/__init__.py': '"""Patient drug event analysis."""\n', 'src/project.py': '"""Reproducible report-level analysis. Run python -m src.project --help."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport joblib\nos.environ.setdefault(\'MPLCONFIGDIR\', str(Path(__file__).resolve().parents[1] / \'outputs\' / \'.matplotlib\'))\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom imblearn.over_sampling import SMOTENC\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,\n    classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score,\n    precision_score, recall_score, roc_auc_score)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.svm import LinearSVC\n\nBASE = Path(__file__).resolve().parents[1]\nTARGET = \'Seriousness_Death_Binary\'\nNUM = [\'Weight\', \'Year\', \'Month\', \'Day\']\nCAT = [\'Country\', \'Gender_Label\', \'Reaction_0\', \'Reaction_1\', \'Reaction_2\',\n       \'Sender_Org\', \'Chemical_Compound\', \'Drug_Indication\', \'Drug_Brand_Name\']\nFEATURES = CAT + NUM\nSOURCE = [\'Country\', \'Gender\', \'Weight\', \'Reaction_0\', \'Reaction_1\', \'Reaction_2\',\n          \'Sender_Org\', \'Seriousness_Death\', \'Seriousness_Disabling\',\n          \'Seriousness_Hospitalization\', \'transmissiondate\', \'Chemical_Compound\',\n          \'Drug_Indication\', \'Drug/Brand name\']\nSERIOUS = [\'Seriousness_Death\', \'Seriousness_Disabling\', \'Seriousness_Hospitalization\']\n\n\ndef folders(base=BASE):\n    for p in [\'data/raw\', \'data/processed\', \'data/synthetic\', \'outputs/plots\',\n              \'outputs/results\', \'models\', \'presentation\', \'notebooks\']:\n        (base / p).mkdir(parents=True, exist_ok=True)\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef read_raw(path):\n    df = pd.read_csv(path, low_memory=False)\n    df.columns = df.columns.str.strip()\n    df = df.loc[:, ~df.columns.str.match(r\'^Unnamed\', case=False)]\n    missing = sorted(set(SOURCE) - set(df.columns))\n    if missing:\n        raise ValueError(f\'Missing required source columns: {missing}\')\n    if df.empty:\n        raise ValueError(\'The input CSV contains no reports.\')\n    return df\n\n\ndef inspect(df, base=BASE):\n    result = base / \'outputs/results\'\n    pd.DataFrame({\'dtype\': df.dtypes.astype(str), \'missing_count\': df.isna().sum(),\n                  \'missing_percent\': df.isna().mean().mul(100),\n                  \'unique_values\': df.nunique(dropna=True)}).to_csv(result / \'source_column_profile.csv\')\n    df[\'Seriousness_Death\'].value_counts(dropna=False).rename_axis(\'source_code\').to_csv(\n        result / \'source_death_codes.csv\', header=[\'reports\'])\n    print(f\'Source shape: {df.shape}\', flush=True)\n\n\ndef clean_data(df):\n    clean = df.rename(columns={\'Drug/Brand name\': \'Drug_Brand_Name\',\n                               \'transmissiondate\': \'Transmission_Date_Raw\'}).copy()\n    for col in clean.select_dtypes(include=[\'object\', \'string\']).columns:\n        clean[col] = clean[col].astype(\'string\').str.strip().replace(\'\', pd.NA)\n    weight = pd.to_numeric(clean[\'Weight\'], errors=\'coerce\')\n    clean[\'Weight\'] = weight.where(weight.between(0, 650, inclusive=\'right\'))\n    clean[\'Gender\'] = pd.to_numeric(clean[\'Gender\'], errors=\'coerce\')\n    clean[\'Gender_Label\'] = clean[\'Gender\'].map({1: \'Male\', 2: \'Female\'}).fillna(\'Unknown\')\n    for col in SERIOUS:\n        clean[col] = pd.to_numeric(clean[col], errors=\'coerce\')\n        clean[col + \'_Binary\'] = clean[col].map({1: 1, 2: 0})\n    date = pd.to_numeric(clean[\'Transmission_Date_Raw\'], errors=\'coerce\')\n    date = date.where(date.mod(1).eq(0)).astype(\'Int64\').astype(\'string\')\n    clean[\'Transmission_Date\'] = pd.to_datetime(date, format=\'%Y%m%d\', errors=\'coerce\')\n    for col, accessor in [(\'Year\', \'year\'), (\'Month\', \'month\'), (\'Day\', \'day\')]:\n        clean[col] = getattr(clean[\'Transmission_Date\'].dt, accessor)\n    for col in CAT:\n        clean[col] = clean[col].fillna(\'Unknown\').astype(str).str.strip()\n    return clean\n\n\ndef save_figure(fig, path):\n    fig.tight_layout()\n    fig.savefig(path, dpi=160, bbox_inches=\'tight\')\n    plt.close(fig)\n\n\ndef bar(series, title, filename, base=BASE, ylabel=\'Reports\'):\n    fig, ax = plt.subplots(figsize=(11, 6))\n    if len(series):\n        ax.bar(range(len(series)), series.values, color=\'#2563a6\')\n        ax.set_xticks(range(len(series)), [str(v) for v in series.index], rotation=45, ha=\'right\')\n    else:\n        ax.text(.5, .5, \'No eligible reports\', ha=\'center\', transform=ax.transAxes)\n    ax.set_title(title)\n    ax.set_ylabel(ylabel)\n    ax.grid(axis=\'y\', alpha=.2)\n    save_figure(fig, base / \'outputs/plots\' / filename)\n\n\ndef known_counts(df, col, n=15):\n    return df.loc[df[col].ne(\'Unknown\'), col].value_counts().head(n)\n\n\ndef eda(df, base=BASE):\n    bar(known_counts(df, \'Gender_Label\'), \'Known patient gender\', \'gender_distribution.png\', base)\n    for col, stem in [(\'Country\', \'countries\'), (\'Drug_Brand_Name\', \'drugs\'), (\'Reaction_0\', \'reactions\')]:\n        counts = known_counts(df, col)\n        counts.to_csv(base / f\'outputs/results/top_{stem}.csv\', header=[\'reports\'])\n        bar(counts, f\'Top reported {stem}\', f\'top_{stem}.png\', base)\n    monthly = df.dropna(subset=[\'Transmission_Date\']).set_index(\'Transmission_Date\').resample(\'ME\').size()\n    monthly.to_csv(base / \'outputs/results/monthly_reports.csv\', header=[\'reports\'])\n    fig, ax = plt.subplots(figsize=(11, 5))\n    if len(monthly) == 1:\n        ax.bar(monthly.index.strftime(\'%Y-%m\'), monthly.values, color=\'#2563a6\')\n    elif len(monthly):\n        ax.plot(monthly.index, monthly.values, marker=\'o\', color=\'#2563a6\')\n    ax.set(title=\'Reports by transmission month\', ylabel=\'Reports\', xlabel=\'Month\')\n    save_figure(fig, base / \'outputs/plots/monthly_reports.png\')\n    seriousness = pd.Series({c.removeprefix(\'Seriousness_\'): int(df[c + \'_Binary\'].eq(1).sum()) for c in SERIOUS})\n    bar(seriousness, \'Flagged serious outcomes (may overlap)\', \'seriousness_counts.png\', base)\n    death = df.loc[df[TARGET].eq(1)]\n    for col, stem in [(\'Drug_Brand_Name\', \'drugs\'), (\'Reaction_0\', \'reactions\')]:\n        bar(known_counts(death, col), f\'Top {stem} in death-associated reports\', f\'top_death_{stem}.png\', base)\n    for country, aliases in {\'USA\': [\'US\', \'USA\', \'UNITED STATES\'], \'Canada\': [\'CA\', \'CANADA\'], \'Brazil\': [\'BR\', \'BRAZIL\']}.items():\n        subset = df.loc[df[\'Country\'].str.upper().isin(aliases)]\n        for col, stem in [(\'Drug_Brand_Name\', \'drugs\'), (\'Reaction_0\', \'reactions\')]:\n            bar(known_counts(subset, col), f\'{country}: top reported {stem}\', f\'top_{stem}_{country.lower()}.png\', base)\n    valid = df.loc[df[\'Drug_Brand_Name\'].ne(\'Unknown\') & df[\'Reaction_0\'].ne(\'Unknown\')]\n    drugs, reactions = known_counts(valid, \'Drug_Brand_Name\').index, known_counts(valid, \'Reaction_0\').index\n    selected = valid.loc[valid[\'Drug_Brand_Name\'].isin(drugs) & valid[\'Reaction_0\'].isin(reactions)]\n    heat = pd.crosstab(selected[\'Drug_Brand_Name\'], selected[\'Reaction_0\']).reindex(index=drugs, columns=reactions, fill_value=0)\n    heat.to_csv(base / \'outputs/results/drug_reaction_cooccurrence.csv\')\n    fig, ax = plt.subplots(figsize=(14, 10))\n    if heat.size:\n        im = ax.imshow(heat.values, aspect=\'auto\', cmap=\'Blues\')\n        fig.colorbar(im, ax=ax, label=\'Co-occurrence count\')\n        ax.set_xticks(range(len(reactions)), reactions, rotation=55, ha=\'right\')\n        ax.set_yticks(range(len(drugs)), drugs)\n    ax.set_title(\'Top drugs and primary reactions: report co-occurrence\')\n    save_figure(fig, base / \'outputs/plots/drug_reaction_heatmap.png\')\n    bar(df[TARGET].map({0: \'Non-death\', 1: \'Death\'}).fillna(\'Unknown\').value_counts(),\n        \'Death outcome coding\', \'death_outcome_distribution.png\', base)\n    print(\'EDA saved.\', flush=True)\n\n\nclass ReportPreparer(BaseEstimator, TransformerMixin):\n    """Learn numeric medians and rare categories exclusively from training rows."""\n    def __init__(self, rare_threshold=25):\n        self.rare_threshold = rare_threshold\n\n    def fit(self, X, y=None):\n        self.medians_ = X[NUM].apply(pd.to_numeric, errors=\'coerce\').median().fillna(0)\n        self.frequent_ = {c: set(X[c].fillna(\'Unknown\').astype(str).value_counts().loc[lambda s: s >= self.rare_threshold].index) for c in CAT}\n        return self\n\n    def transform(self, X):\n        result = X[FEATURES].copy()\n        result[NUM] = result[NUM].apply(pd.to_numeric, errors=\'coerce\').fillna(self.medians_)\n        for c in CAT:\n            text = result[c].fillna(\'Unknown\').astype(str)\n            result[c] = text.where(text.isin(self.frequent_[c]), \'__RARE__\')\n        return result\n\n\ndef encoder():\n    return ColumnTransformer([\n        (\'num\', Pipeline([(\'imputer\', SimpleImputer(strategy=\'median\', keep_empty_features=True)),\n                          (\'scaler\', StandardScaler())]), NUM),\n        (\'cat\', OneHotEncoder(handle_unknown=\'ignore\', sparse_output=True, dtype=np.float32), CAT),\n    ], sparse_threshold=1.0)\n\n\ndef algorithms(seed=42):\n    return {\n        \'Logistic Regression\': LogisticRegression(max_iter=500, solver=\'saga\', random_state=seed),\n        \'Random Forest\': RandomForestClassifier(n_estimators=120, max_depth=25, random_state=seed, n_jobs=-1),\n        \'Linear SVM\': LinearSVC(random_state=seed, max_iter=5000, dual=\'auto\'),\n        \'KNN\': KNeighborsClassifier(n_neighbors=5, algorithm=\'brute\', n_jobs=-1),\n    }\n\n\ndef subset(df, size, seed):\n    model_df = df.loc[df[TARGET].notna()].copy()\n    model_df[TARGET] = model_df[TARGET].astype(int)\n    if model_df[TARGET].nunique() != 2:\n        raise ValueError(\'Supervised training needs both known death and non-death outcomes.\')\n    if size < 20:\n        raise ValueError(\'Use a sample size of at least 20.\')\n    if len(model_df) > size:\n        model_df, _ = train_test_split(model_df, train_size=size, stratify=model_df[TARGET], random_state=seed)\n    if model_df[TARGET].value_counts().min() < 3:\n        raise ValueError(\'Too few examples in the minority class; increase sample size.\')\n    return model_df\n\n\ndef train(df, run, size, base=BASE, seed=42, rare_threshold=25):\n    model_df = subset(df, size, seed)\n    X_train, X_test, y_train, y_test = train_test_split(model_df[FEATURES], model_df[TARGET],\n        test_size=.2, stratify=model_df[TARGET], random_state=seed)\n    # Persist source row positions to audit train/test disjointness. No report identifiers are available.\n    split = pd.DataFrame({\'source_row\': list(X_train.index) + list(X_test.index),\n                           \'partition\': [\'train\'] * len(X_train) + [\'test\'] * len(X_test)})\n    split.to_csv(base / f\'outputs/results/{run.lower()}_split_rows.csv\', index=False)\n    prepare = ReportPreparer(rare_threshold).fit(X_train)\n    training, testing = prepare.transform(X_train), prepare.transform(X_test)\n    before = y_train.value_counts().sort_index().to_dict()\n    resample_seconds = 0.0\n    numeric_scale = None\n    if run == \'SMOTENC\':\n        minority = int(y_train.value_counts().min())\n        if minority < 2:\n            raise ValueError(\'SMOTENC requires at least two minority training examples.\')\n        # Standardize continuous variables before nearest-neighbor distances.\n        numeric_scale = StandardScaler().fit(training[NUM])\n        training[NUM] = numeric_scale.transform(training[NUM])\n        testing[NUM] = numeric_scale.transform(testing[NUM])\n        sampler = SMOTENC(categorical_features=CAT, random_state=seed, k_neighbors=min(5, minority - 1))\n        start = time.perf_counter()\n        training, y_train = sampler.fit_resample(training, y_train)\n        resample_seconds = time.perf_counter() - start\n    after = pd.Series(y_train).value_counts().sort_index().to_dict()\n    enc = encoder()\n    start = time.perf_counter()\n    encoded_train = enc.fit_transform(training)\n    encoded_test = enc.transform(testing)\n    preprocessing_seconds = time.perf_counter() - start\n    results = []\n    reports, manifest = {}, {}\n    for name, model in {\'Majority Dummy\': DummyClassifier(strategy=\'most_frequent\'), **algorithms(seed)}.items():\n        print(f\'{run}: fitting {name} on {encoded_train.shape}\', flush=True)\n        start = time.perf_counter()\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter(\'always\', ConvergenceWarning)\n            model.fit(encoded_train, y_train)\n        fit_seconds = time.perf_counter() - start\n        start = time.perf_counter()\n        pred = model.predict(encoded_test)\n        predict_seconds = time.perf_counter() - start\n        scores = model.predict_proba(encoded_test)[:, 1] if hasattr(model, \'predict_proba\') else model.decision_function(encoded_test)\n        cm = confusion_matrix(y_test, pred, labels=[0, 1])\n        tn, fp, fn, tp = cm.ravel()\n        result = dict(run=run, model=name, accuracy=accuracy_score(y_test, pred),\n            balanced_accuracy=balanced_accuracy_score(y_test, pred), precision_death=precision_score(y_test, pred, zero_division=0),\n            recall_death=recall_score(y_test, pred, zero_division=0), f1_death=f1_score(y_test, pred, zero_division=0),\n            roc_auc=roc_auc_score(y_test, scores), average_precision=average_precision_score(y_test, scores),\n            train_seconds=fit_seconds, predict_seconds=predict_seconds, preprocessing_seconds=preprocessing_seconds,\n            resample_seconds=resample_seconds, sample_rows=len(model_df), train_rows=len(y_train), test_rows=len(y_test),\n            encoded_features=encoded_train.shape[1], tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),\n            convergence_warning=any(issubclass(w.category, ConvergenceWarning) for w in caught))\n        results.append(result)\n        stem = name.lower().replace(\' \', \'_\')\n        reports[name] = classification_report(y_test, pred, output_dict=True, zero_division=0)\n        fig, ax = plt.subplots(figsize=(6, 5))\n        ConfusionMatrixDisplay(cm, display_labels=[\'Non-death\', \'Death\']).plot(ax=ax, values_format=\'d\', cmap=\'Blues\', colorbar=False)\n        ax.set_title(f\'{run}: {name}\')\n        save_figure(fig, base / f\'outputs/plots/{run.lower()}_{stem}_confusion_matrix.png\')\n        pd.DataFrame({\'source_row\': X_test.index, \'actual\': y_test.to_numpy(), \'predicted\': pred,\n                      \'score\': scores}).to_csv(base / f\'outputs/results/{run.lower()}_{stem}_predictions.csv\', index=False)\n        if name != \'Majority Dummy\':\n            joblib.dump({\'prepare\': prepare, \'numeric_scale\': numeric_scale, \'encoder\': enc,\n                         \'model\': model, \'features\': FEATURES, \'run\': run}, base / f\'models/{run.lower()}_{stem}.joblib\', compress=3)\n        manifest[name] = [str(w.message) for w in caught]\n        print(f\'{name}: accuracy={result["accuracy"]:.4f}, recall={result["recall_death"]:.4f}, F1={result["f1_death"]:.4f}\', flush=True)\n        pd.DataFrame(results).to_csv(base / f\'outputs/results/{run.lower()}_metrics.csv\', index=False)\n    write_json(base / f\'outputs/results/{run.lower()}_classification_reports.json\', reports)\n    write_json(base / f\'outputs/results/{run.lower()}_run.json\', dict(seed=seed, requested_sample=size,\n        actual_sample=len(model_df), test_fraction=.2, rare_threshold=rare_threshold,\n        train_before_resampling=before, train_after_resampling=after, test_counts=y_test.value_counts().to_dict(),\n        warnings=manifest, algorithms={k: v.get_params() for k, v in algorithms(seed).items()}))\n    return pd.DataFrame(results)\n\n\ndef summary(raw, clean, base=BASE):\n    values = {\'source_rows\': len(raw), \'source_columns\': len(raw.columns), \'known_death_outcomes\': int(clean[TARGET].notna().sum()),\n        \'death_reports\': int(clean[TARGET].eq(1).sum()), \'non_death_reports\': int(clean[TARGET].eq(0).sum()),\n        \'unknown_death_outcomes\': int(clean[TARGET].isna().sum()), \'exact_duplicate_source_rows\': int(raw.duplicated().sum()),\n        \'hospitalization_flagged\': int(clean[\'Seriousness_Hospitalization_Binary\'].eq(1).sum()),\n        \'disabling_flagged\': int(clean[\'Seriousness_Disabling_Binary\'].eq(1).sum()),\n        \'invalid_or_missing_weight\': int(clean[\'Weight\'].isna().sum()),\n        \'invalid_or_missing_date\': int(clean[\'Transmission_Date\'].isna().sum())}\n    pd.Series(values).rename_axis(\'metric\').to_csv(base / \'outputs/results/dataset_summary.csv\', header=[\'value\'])\n    clean[TARGET].value_counts(dropna=False).rename_axis(\'death_binary\').to_csv(base / \'outputs/results/class_distribution.csv\', header=[\'reports\'])\n    return values\n\n\ndef compare(base=BASE):\n    result = base / \'outputs/results\'\n    paths = [result / \'baseline_metrics.csv\', result / \'smotenc_metrics.csv\']\n    if not all(p.exists() for p in paths):\n        raise FileNotFoundError(\'Run both baseline and SMOTENC training before building comparisons.\')\n    comparison = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)\n    comparison.to_csv(result / \'model_comparison.csv\', index=False)\n    for metric, title, stem in [(\'accuracy\', \'Accuracy\', \'model_accuracy\'), (\'recall_death\', \'Death recall\', \'death_recall\'),\n                                 (\'f1_death\', \'Death F1\', \'death_f1\'), (\'precision_death\', \'Death precision\', \'death_precision\')]:\n        pivot = comparison.loc[comparison.model.ne(\'Majority Dummy\')].pivot(index=\'model\', columns=\'run\', values=metric)\n        ax = pivot.plot.bar(figsize=(10, 6), color=[\'#2563a6\', \'#d98232\'])\n        ax.set(title=f\'{title}: baseline and SMOTENC (different sample sizes)\', ylim=(0, 1.05), ylabel=title, xlabel=\'\')\n        plt.setp(ax.get_xticklabels(), rotation=25, ha=\'right\')\n        save_figure(ax.figure, base / f\'outputs/plots/{stem}_comparison.png\')\n    dataset = pd.read_csv(result / \'dataset_summary.csv\')\n    rows = [\'# Results\', \'\', \'Measured from the supplied merged CSV. Historical guide figures are recorded separately in presentation/README.md.\', \'\',\n        \'## Dataset\', \'\', \'| Metric | Value |\', \'| --- | ---: |\']\n    rows += [f\'| {r.metric} | {int(r.value):,} |\' for r in dataset.itertuples()]\n    rows += [\'\', \'## Evaluation\', \'\', \'| Run | Model | Accuracy | Death precision | Death recall | Death F1 | Average precision | Fit seconds |\',\n        \'| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |\']\n    for r in comparison.itertuples():\n        rows.append(f\'| {r.run} | {r.model} | {r.accuracy:.4f} | {r.precision_death:.4f} | {r.recall_death:.4f} | {r.f1_death:.4f} | {r.average_precision:.4f} | {r.train_seconds:.2f} |\')\n    best = comparison.loc[(comparison.run == \'Baseline\') & comparison.model.ne(\'Majority Dummy\')].sort_values(\'f1_death\', ascending=False).iloc[0]\n    rows += [\'\', \'## Measured observations\', \'\',\n        f\'{best.model} has the highest baseline death F1 in this run ({best.f1_death:.4f}), with accuracy {best.accuracy:.2%}, death precision {best.precision_death:.2%}, and death recall {best.recall_death:.2%}. This ranking describes this particular split and is not a clinical model recommendation.\', \'\']\n    for name in algorithms():\n        pair = comparison.loc[comparison.model.eq(name)].set_index(\'run\')\n        delta = pair.loc[\'SMOTENC\', \'recall_death\'] - pair.loc[\'Baseline\', \'recall_death\']\n        rows.append(f\'{name}: death recall changes by {delta * 100:+.2f} percentage points between the two runs. Baseline F1 is {pair.loc["Baseline", "f1_death"]:.4f}; SMOTENC F1 is {pair.loc["SMOTENC", "f1_death"]:.4f}.\')\n        rows.append(\'\')\n    rows += [\'\', \'## Interpretation and limits\', \'\',\n        \'Baseline uses up to 65,000 known-target rows; SMOTENC uses up to 15,000 before resampling. Each run has its own untouched stratified 20% test split. Differences combine sample-size effects with balancing effects, so this is not a controlled estimate of the effect of SMOTENC. For a controlled comparison, rerun both with the same sample size and seed.\', \'\',\n        \'Rare categories, imputation, scaling and encoding are fitted on training rows only. SMOTENC sees training rows only. Numeric fields are standardized before its neighbor search. Hospitalization, disability and death codes are excluded from predictors.\', \'\',\n        \'These are retrospective classifications of reports with known outcome codes, not prospective predictions of patient mortality or evidence that a drug caused death. Reactions may already describe an outcome. Spontaneous report counts are not drug incidence rates. The merged file has no report identifier: repeated reports, follow-up versions and shared patients cannot reliably be grouped, and overlapping records may inflate random-split performance. Unknown outcomes are excluded and may introduce selection bias. No clinical validation, confidence intervals or external test cohort is provided.\', \'\',\n        \'Exact source duplicates are counted but retained to preserve the documented source experiment. Review deduplication and report-level identifiers before making substantive conclusions. All convergence warnings are saved in each run JSON; warning flags also appear in metrics CSVs. Models are trusted local joblib artifacts; do not load untrusted joblib files.\', \'\']\n    (base / \'RESULTS.md\').write_text(\'\\n\'.join(rows), encoding=\'utf-8\')\n    return comparison\n\n\ndef synthetic(raw, output, target_bytes, chunk_rows=100000, seed=42):\n    if target_bytes <= 0 or chunk_rows <= 0:\n        raise ValueError(\'Target bytes and chunk rows must be positive.\')\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(f\'Refusing to append to an existing experiment: {output}\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    rng, total = np.random.default_rng(seed), 0\n    while not output.exists() or output.stat().st_size < target_bytes:\n        chunk = raw.iloc[rng.integers(0, len(raw), size=chunk_rows)].reset_index(drop=True).copy()\n        weight = pd.to_numeric(chunk[\'Weight\'], errors=\'coerce\').astype(float)\n        valid = weight.between(0, 650, inclusive=\'right\')\n        weight.loc[valid] = (weight.loc[valid] + rng.normal(0, 1, int(valid.sum()))).clip(.1, 650)\n        chunk[\'Weight\'] = weight\n        chunk.to_csv(output, mode=\'a\', index=False, header=not output.exists())\n        total += len(chunk)\n    write_json(output.with_suffix(\'.manifest.json\'), {\'purpose\': \'Bootstrapped scalability data, not additional real reports\',\n        \'rows\': total, \'bytes\': output.stat().st_size, \'target_bytes\': target_bytes, \'seed\': seed})\n    return total\n\n\ndef provenance(path, args, base=BASE):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as file:\n        for block in iter(lambda: file.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    write_json(base / \'outputs/results/provenance.json\', {\'input_filename\': Path(path).name,\n        \'input_sha256\': digest.hexdigest(), \'input_bytes\': Path(path).stat().st_size,\n        \'python\': platform.python_version(), \'packages\': {p: importlib.metadata.version(p)\n        for p in [\'pandas\', \'numpy\', \'matplotlib\', \'scikit-learn\', \'imbalanced-learn\', \'joblib\']},\n        \'seed\': args.seed, \'baseline_sample\': args.baseline_sample, \'smotenc_sample\': args.smotenc_sample,\n        \'input_kind\': \'User-supplied merged adverse-event CSV; not fetched directly from OpenFDA\'})\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'step\', choices=[\'all\', \'inspect\', \'clean\', \'eda\', \'baseline\', \'smotenc\', \'synthetic\', \'outputs\'])\n    parser.add_argument(\'--data\', type=Path, default=BASE / \'data/raw/2025_merged.csv\')\n    parser.add_argument(\'--baseline-sample\', type=int, default=65000)\n    parser.add_argument(\'--smotenc-sample\', type=int, default=15000)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--rare-threshold\', type=int, default=25)\n    parser.add_argument(\'--synthetic-bytes\', type=int, default=1000000000)\n    parser.add_argument(\'--chunk-rows\', type=int, default=100000)\n    parser.add_argument(\'--synthetic-output\', type=Path, default=BASE / \'data/synthetic/synthetic_drug_event_1GB.csv\')\n    args = parser.parse_args(argv)\n    folders()\n    if args.step == \'outputs\':\n        compare()\n        return\n    raw = read_raw(args.data)\n    if args.step == \'synthetic\':\n        print(f\'Generated {synthetic(raw, args.synthetic_output, args.synthetic_bytes, args.chunk_rows, args.seed):,} synthetic rows.\')\n        return\n    if args.step in [\'all\', \'inspect\']:\n        inspect(raw)\n    clean = clean_data(raw)\n    summary(raw, clean)\n    if args.step in [\'all\', \'clean\']:\n        clean.to_csv(BASE / \'data/processed/2025_merged_clean.csv\', index=False)\n        raw.sample(n=min(200, len(raw)), random_state=args.seed).to_csv(BASE / \'data/sample_patient_drug_events.csv\', index=False)\n        print(\'Cleaned data saved.\', flush=True)\n    if args.step in [\'all\', \'eda\']:\n        eda(clean)\n    if args.step in [\'all\', \'baseline\']:\n        train(clean, \'Baseline\', args.baseline_sample, seed=args.seed, rare_threshold=args.rare_threshold)\n    if args.step in [\'all\', \'smotenc\']:\n        train(clean, \'SMOTENC\', args.smotenc_sample, seed=args.seed, rare_threshold=args.rare_threshold)\n    if args.step == \'all\':\n        compare()\n        provenance(args.data, args)\n    print(f\'Finished {args.step}. Outputs: {BASE / "outputs"}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    # Import under the canonical package name so saved estimators can be loaded\n    # by a different process (pickle must not refer to __main__.ReportPreparer).\n    from src.project import main as cli_main\n    cli_main()\n', 'src/predict.py': '"""Classify reports using a trusted saved local model."""\nimport argparse\nfrom pathlib import Path\nimport joblib\nimport pandas as pd\nfrom src.project import clean_data, read_raw, FEATURES, NUM\n\n\ndef predict(bundle, raw):\n    clean = clean_data(raw)\n    prepared = bundle[\'prepare\'].transform(clean[FEATURES])\n    if bundle[\'numeric_scale\'] is not None:\n        prepared[NUM] = bundle[\'numeric_scale\'].transform(prepared[NUM])\n    encoded = bundle[\'encoder\'].transform(prepared)\n    output = pd.DataFrame({\'source_row\': raw.index, \'predicted_death_report\': bundle[\'model\'].predict(encoded)})\n    if hasattr(bundle[\'model\'], \'predict_proba\'):\n        output[\'model_score\'] = bundle[\'model\'].predict_proba(encoded)[:, 1]\n    else:\n        output[\'model_score\'] = bundle[\'model\'].decision_function(encoded)\n    return output\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--model\', required=True, type=Path)\n    p.add_argument(\'--data\', required=True, type=Path)\n    p.add_argument(\'--output\', type=Path, default=Path(\'outputs/results/new_predictions.csv\'))\n    a = p.parse_args()\n    a.output.parent.mkdir(parents=True, exist_ok=True)\n    predict(joblib.load(a.model), read_raw(a.data)).to_csv(a.output, index=False)\n    print(a.output)\n'}
for name, source in SOURCES.items():
    path = BASE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(BASE))
from src.project import *
folders(BASE)
print(BASE)

## Upload the real merged CSV

Select exactly one source CSV. For Google Drive, replace this cell with a Drive mount and set DATA_PATH to your file.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one merged CSV")
name = next(iter(uploaded))
DATA_PATH = BASE / "data/raw/2025_merged.csv"
DATA_PATH.write_bytes(uploaded[name])
del uploaded
raw = read_raw(DATA_PATH)
inspect(raw, BASE)
display(raw.head())
display(pd.read_csv(BASE / "outputs/results/source_column_profile.csv"))

## Clean and summarize

Weight outside (0,650] becomes missing. Invalid dates become missing. Only death codes 1 and 2 become known targets; no Age column is invented.

In [ ]:
clean = clean_data(raw)
clean.to_csv(BASE / "data/processed/2025_merged_clean.csv", index=False)
raw.sample(min(200, len(raw)), random_state=42).to_csv(BASE / "data/sample_patient_drug_events.csv", index=False)
display(pd.Series(summary(raw, clean, BASE)).to_frame("value"))

## Exploratory analysis

Report counts and co-occurrence describe reporting patterns; they do not establish drug risk or causation.

In [ ]:
eda(clean, BASE)
from IPython.display import Image, display
for name in ["top_countries.png", "drug_reaction_heatmap.png", "death_outcome_distribution.png"]:
    display(Image(filename=str(BASE / "outputs/plots" / name)))

## Baseline models

Training-only category grouping, median imputation, scaling and encoding. The dummy reference helps interpret accuracy. KNN may take several minutes.

In [ ]:
MODEL_SAMPLE_SIZE = 65000
SMOTENC_SAMPLE_SIZE = 15000
SEED = 42
baseline_results = train(clean, "Baseline", MODEL_SAMPLE_SIZE, BASE, SEED)
display(baseline_results)

## Training-only SMOTENC

The original test split remains unchanged. Continuous fields are standardized before neighbor search. This smaller sample follows the guide; use the same sample size in both runs for a controlled comparison.

In [ ]:
smote_results = train(clean, "SMOTENC", SMOTENC_SAMPLE_SIZE, BASE, SEED)
display(smote_results)

## Compare and save

Read generated RESULTS.md for measured metrics and limitations. Random splits may share related or duplicate reports because no report identifier is available.

In [ ]:
comparison = compare(BASE)
from types import SimpleNamespace
provenance(DATA_PATH, SimpleNamespace(seed=SEED, baseline_sample=MODEL_SAMPLE_SIZE, smotenc_sample=SMOTENC_SAMPLE_SIZE), BASE)
display(comparison)
for name in ["model_accuracy_comparison.png", "death_recall_comparison.png", "death_f1_comparison.png"]:
    display(Image(filename=str(BASE / "outputs/plots" / name)))
print((BASE / "RESULTS.md").read_text())

## Optional scalability experiment

Disabled by default. This bootstraps whole rows and modifies only weight; it creates no new real clinical evidence.

In [ ]:
RUN_SYNTHETIC_EXPERIMENT = False
if RUN_SYNTHETIC_EXPERIMENT:
    synthetic(raw, BASE / "data/synthetic/synthetic_drug_event_1GB.csv", 1_000_000_000)

## Download results

The archive includes results, plots, notebook-source modules and trusted model bundles. Full raw, processed and synthetic data are excluded.

In [ ]:
import zipfile
ZIP_PATH = Path("/content/patient-drug-event-analysis-results.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ["outputs", "models", "src"]:
        for path in (BASE / folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(BASE))
    archive.write(BASE / "RESULTS.md", "RESULTS.md")
files.download(str(ZIP_PATH))